In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from joblib import Parallel, delayed
from sklearn.neural_network import MLPClassifier

# Load the dataset
# Note: You should run this on your local environment as I don't have access to your file path.
data = pd.read_excel(# enter file path)

# Extract the predictors and outcome
X = data.drop('RRI', axis=1)
Y = data['RRI']

# List of alpha values to try
alpha_values = np.logspace(-1, -3, 20)

# List of classifiers along with their names
classifiers = [
    ("Decision Tree", DecisionTreeClassifier(random_state=42)),
    ("Random Forest", RandomForestClassifier(random_state=42)),
    ("SVM", SVC(probability=True, random_state=42)),
    ("KNN", KNeighborsClassifier()),
    ("Naïve Bayes", GaussianNB()),
    ("Adaboost", AdaBoostClassifier(random_state=42)),
    ("Gradient Boosting", GradientBoostingClassifier(random_state=42)),
    ('MLP', MLPClassifier(max_iter=1000, random_state=42))  # Setting max_iter to 1000 to ensure convergence
]

# Dictionary to store the best AUC, feature combination, and alpha for each classifier
best_results_by_classifier = {name: (0, None, None) for name, _ in classifiers}

def compute_best_result(alpha, classifier_name, model, X, Y):
    """
    Compute the best ROC AUC score for a given alpha and classifier.

    Parameters:
    - alpha: Regularization strength for Lasso (inverse of C).
    - classifier_name: Name of the classifier.
    - model: The classifier instance.
    - X: All features.
    - Y: All labels.

    Returns:
    - Tuple containing classifier name, best AUC, best feature indices, and alpha.
    """
    # Initialize L1-penalized Logistic Regression (Logistic Lasso)
    lasso = LogisticRegression(penalty='l1', C=1/alpha, solver='saga', max_iter=10000, random_state=42)
    
    # Fit the Lasso model on the entire dataset
    lasso.fit(X, Y)
    
    # Get the coefficients (they are returned as a 2D array, so flatten it)
    coefficients = lasso.coef_[0]
    
    # Extract non-zero coefficients and sort them by absolute value
    non_zero_coefficients = sorted(
        [(index, coef) for index, coef in enumerate(coefficients) if coef != 0],
        key=lambda x: abs(x[1]),
        reverse=True
    )
    
    max_auc_for_alpha = 0
    best_feature_combination_for_alpha = None
    
    for i in range(1, len(non_zero_coefficients) + 1):
        top_features_indices = [index for index, _ in non_zero_coefficients[:i]]
        X_reduced = X.iloc[:, top_features_indices]
        skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
        aucs = []
        for train_index, test_index in skf.split(X_reduced, Y):
            X_train_fold, X_test_fold = X_reduced.iloc[train_index], X_reduced.iloc[test_index]
            y_train_fold, y_test_fold = Y.iloc[train_index], Y.iloc[test_index]
            model.fit(X_train_fold, y_train_fold)
            if hasattr(model, "predict_proba"):
                y_pred_probs_fold = model.predict_proba(X_test_fold)[:, 1]
            else:
                # For classifiers that do not have predict_proba, use decision_function
                y_pred_probs_fold = model.decision_function(X_test_fold)
                # Scale the decision function to [0,1] using min-max scaling
                y_pred_probs_fold = (y_pred_probs_fold - y_pred_probs_fold.min()) / (y_pred_probs_fold.max() - y_pred_probs_fold.min() + 1e-8)
            aucs.append(roc_auc_score(y_test_fold, y_pred_probs_fold))

        mean_auc = np.mean(aucs)
        if mean_auc > max_auc_for_alpha:
            max_auc_for_alpha = mean_auc
            best_feature_combination_for_alpha = top_features_indices

    return (classifier_name, max_auc_for_alpha, best_feature_combination_for_alpha, alpha)

# Create a list of all (alpha, classifier) pairs with data
tasks = [(alpha, name, model, X, Y) for alpha in alpha_values for name, model in classifiers]

# Parallelize the computation across available cores
results = Parallel(n_jobs=-1)(
    delayed(compute_best_result)(alpha, name, model, X, Y) for alpha, name, model, X, Y in tasks
)

# Aggregate results to find the best for each classifier
for classifier_name, auc, features, alpha in results:
    current_best_auc, _, _ = best_results_by_classifier[classifier_name]
    if auc > current_best_auc:
        best_results_by_classifier[classifier_name] = (auc, features, alpha)

# Note: The final results are stored in the 'best_results_by_classifier' dictionary.
best_results_by_classifier